# CCLM+CIRA Lab Project (Colab)

This notebook is a Google Colab workflow for running training and evaluation end-to-end.

## 1) Clone repository (if needed)

If this notebook was opened from your cloned repo already, this step can be skipped.

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/<your-username>/cognitive-constraint.git"  # change this
REPO_DIR = Path("/content/cognitive-constraint")

if not REPO_DIR.exists():
    if "<your-username>" in REPO_URL:
        raise ValueError("Please update REPO_URL with your actual GitHub repository URL.")
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/cognitive-constraint
print("Working directory:", os.getcwd())

## 2) Install dependencies

In [ ]:
%cd /content/cognitive-constraint
!python -m pip install --upgrade pip
!pip install -r requirements.txt

## 3) Verify project files

In [ ]:
from pathlib import Path

required = [
    "configs/base.yaml",
    "data/cira_lab_dataset.json",
    "scripts/train.py",
    "scripts/evaluate.py",
]

missing = [p for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

print("All required files are present.")

## 4) Train model

In [ ]:
%cd /content/cognitive-constraint
!python scripts/train.py --config configs/base.yaml --dataset data/cira_lab_dataset.json --output outputs/run_01

## 5) Evaluate checkpoint

In [ ]:
%cd /content/cognitive-constraint
!python scripts/evaluate.py --checkpoint outputs/run_01/best_model.pt --dataset data/cira_lab_dataset.json

## 6) Inspect generated artifacts

In [ ]:
import json
from pathlib import Path

output_dir = Path("outputs/run_01")
artifacts = [
    output_dir / "best_model.pt",
    output_dir / "history.json",
    output_dir / "metrics.json",
]

for p in artifacts:
    print(f"{p}:", "OK" if p.exists() else "MISSING")

if (output_dir / "metrics.json").exists():
    metrics = json.loads((output_dir / "metrics.json").read_text(encoding="utf-8"))
    print("\nMetrics summary:")
    print(json.dumps(metrics, indent=2))

## 7) Plot training curve

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

hist_path = Path("outputs/run_01/history.json")
if not hist_path.exists():
    raise FileNotFoundError("Run training first: outputs/run_01/history.json not found.")

history = json.loads(hist_path.read_text(encoding="utf-8"))
epochs = [int(row["epoch"]) for row in history]
train_loss = [row["train_loss"] for row in history]
val_loss = [row["val_loss"] for row in history]
val_acc = [row["val_accuracy"] for row in history]

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, train_loss, label="train_loss")
plt.plot(epochs, val_loss, label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curves")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, val_acc, label="val_accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy")
plt.legend()

plt.tight_layout()
plt.show()